# Hybrid Residual V1

Model này giữ ARX Search V1 làm backbone, rồi train model thứ hai để học phần lỗi còn lại:

`y_hybrid = y_arx_sim + shrink * residual_model(features)`

Residual model không dùng `y_true` của validation/test, chỉ dùng ARX simulated output và input/setpoint đã biết. Hệ số `shrink` được chọn bằng validation `FIT_sim`.


In [1]:
from pathlib import Path
import json
import sys
import time

import numpy as np
import pandas as pd
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

WORK_DIR = Path.cwd()
PROJECT_ROOT = WORK_DIR.parent if WORK_DIR.name == "Hybrid_ARX_NARX" else WORK_DIR
OUT_DIR = PROJECT_ROOT / "Hybrid_ARX_NARX"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from arx_pipeline import (
    DataConfig,
    SplitConfig,
    ModelConfig,
    load_or_generate_data,
    split_time_series,
    build_regression_matrix,
    estimate_ols,
    simulate_arx,
    compute_metrics,
)
from narx_pipeline import (
    build_augmented_df,
    fit_zscore_stats,
    apply_zscore,
    inverse_zscore_y,
    scaled_clip_bounds,
)

BASELINE_INPUT_COLS = ("Temperature", "Humidity", "Light", "Drip", "Mist", "Fan")
AUGMENTED_INPUT_COLS = (
    *BASELINE_INPUT_COLS,
    "Light_log",
    "Temp_x_Humi",
    "Temp_x_Light",
    "Humi_x_Light",
    "SP_Center",
    "SP_Width",
    "Month_sin",
    "Month_cos",
    "Season_sin",
    "Season_cos",
)
SCALE_COLS = ("Soil_Moisture", *AUGMENTED_INPUT_COLS)
SHRINK_CANDIDATES = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 1.0]


## 1. Data, ARX backbone


In [2]:
df_full, _, data_source = load_or_generate_data(
    DataConfig(
        csv_path=PROJECT_ROOT / "greenhouse_data.csv",
        generator_script_path=PROJECT_ROOT / "data_generator.py",
        force_regenerate_from_script=False,
        auto_save_generated_csv=True,
    )
)
df_aug = build_augmented_df(df_full)
df_train, df_val, df_test = split_time_series(df_aug, SplitConfig(train_ratio=0.60, val_ratio=0.20))

scale_stats = fit_zscore_stats(df_train, SCALE_COLS)
clip_bounds_real, clip_bounds_scaled = scaled_clip_bounds(df_train, scale_stats, (0.01, 0.99))
df_train_z = apply_zscore(df_train, scale_stats)
df_val_z = apply_zscore(df_val, scale_stats)
df_test_z = apply_zscore(df_test, scale_stats)

ARX_CONFIG = ModelConfig(
    na=1,
    nb=5,
    nk=2,
    include_intercept=True,
    input_cols=AUGMENTED_INPUT_COLS,
    simulation_clip=clip_bounds_scaled,
)

x_train_arx, y_train_arx = build_regression_matrix(df_train_z, ARX_CONFIG)
theta_arx, _, sigma2_arx = estimate_ols(x_train_arx, y_train_arx)

y_arx_train, y_true_train = simulate_arx(df_train_z, theta_arx, ARX_CONFIG)
y_arx_val, y_true_val = simulate_arx(df_val_z, theta_arx, ARX_CONFIG)
y_arx_test, y_true_test = simulate_arx(df_test_z, theta_arx, ARX_CONFIG)

arx_val_metrics = compute_metrics(
    inverse_zscore_y(y_true_val, scale_stats),
    inverse_zscore_y(y_arx_val, scale_stats),
    ARX_CONFIG.na + len(AUGMENTED_INPUT_COLS) * ARX_CONFIG.nb + 1,
)
arx_test_metrics = compute_metrics(
    inverse_zscore_y(y_true_test, scale_stats),
    inverse_zscore_y(y_arx_test, scale_stats),
    ARX_CONFIG.na + len(AUGMENTED_INPUT_COLS) * ARX_CONFIG.nb + 1,
)

pd.Series({
    "data_source": data_source,
    "arx_order": "(1,5,2)",
    "arx_val_FIT_sim": arx_val_metrics["FIT"],
    "arx_test_FIT_sim": arx_test_metrics["FIT"],
    "arx_test_RMSE_sim": arx_test_metrics["RMSE"],
})


data_source          CSV:greenhouse_data.csv
arx_order                            (1,5,2)
arx_val_FIT_sim                    68.955943
arx_test_FIT_sim                   66.833743
arx_test_RMSE_sim                   0.966113
dtype: object

## 2. Residual feature matrix


In [3]:
def build_residual_features(df_z: pd.DataFrame, y_arx_sim: np.ndarray, cfg: ModelConfig) -> np.ndarray:
    lag = max(cfg.na, cfg.nb + cfg.nk - 1)
    y_arx_full = df_z[cfg.output_col].astype(float).to_numpy().copy()
    y_arx_full[lag:] = y_arx_sim
    rows = []
    for idx, t in enumerate(range(lag, len(df_z))):
        row = [float(y_arx_sim[idx])]
        for y_lag in range(1, 4):
            row.append(float(y_arx_full[t - y_lag]))
        for col in AUGMENTED_INPUT_COLS:
            values = df_z[col].astype(float).to_numpy()
            row.append(float(values[t]))
            row.append(float(values[t - 1]))
            row.append(float(values[t - 2]))
        rows.append(row)
    return np.asarray(rows, dtype=float)


def evaluate_hybrid(y_true_z: np.ndarray, y_arx_z: np.ndarray, correction_z: np.ndarray, shrink: float) -> dict[str, float]:
    y_hybrid_z = y_arx_z + shrink * correction_z
    y_hybrid_z = np.clip(y_hybrid_z, clip_bounds_scaled[0], clip_bounds_scaled[1])
    return compute_metrics(
        inverse_zscore_y(y_true_z, scale_stats),
        inverse_zscore_y(y_hybrid_z, scale_stats),
        0,
    )


x_res_train = build_residual_features(df_train_z, y_arx_train, ARX_CONFIG)
x_res_val = build_residual_features(df_val_z, y_arx_val, ARX_CONFIG)
x_res_test = build_residual_features(df_test_z, y_arx_test, ARX_CONFIG)
y_res_train = y_true_train - y_arx_train

x_res_train.shape, x_res_val.shape, x_res_test.shape


((63066, 52), (21018, 52), (21018, 52))

## 3. Train residual candidates


In [4]:
MODEL_CANDIDATES = [
    ("ridge_0.1", make_pipeline(StandardScaler(), Ridge(alpha=0.1))),
    ("ridge_1", make_pipeline(StandardScaler(), Ridge(alpha=1.0))),
    ("hgb_31", HistGradientBoostingRegressor(
        max_iter=180,
        learning_rate=0.05,
        max_leaf_nodes=31,
        l2_regularization=0.0,
        early_stopping=True,
        validation_fraction=0.15,
        n_iter_no_change=15,
        random_state=52,
    )),
    ("hgb_15_l2", HistGradientBoostingRegressor(
        max_iter=220,
        learning_rate=0.04,
        max_leaf_nodes=15,
        l2_regularization=1e-4,
        early_stopping=True,
        validation_fraction=0.15,
        n_iter_no_change=20,
        random_state=53,
    )),
    ("extra_trees_leaf5", ExtraTreesRegressor(
        n_estimators=160,
        max_features=0.7,
        min_samples_leaf=5,
        n_jobs=-1,
        random_state=54,
    )),
    ("random_forest_leaf5", RandomForestRegressor(
        n_estimators=160,
        max_features=0.7,
        min_samples_leaf=5,
        n_jobs=-1,
        random_state=55,
    )),
]

rows = []
for model_name, model in MODEL_CANDIDATES:
    start = time.time()
    model.fit(x_res_train, y_res_train)
    train_seconds = time.time() - start
    corr_val = model.predict(x_res_val)
    corr_test = model.predict(x_res_test)

    for shrink in SHRINK_CANDIDATES:
        val_metrics = evaluate_hybrid(y_true_val, y_arx_val, corr_val, shrink)
        test_metrics = evaluate_hybrid(y_true_test, y_arx_test, corr_test, shrink)
        rows.append({
            "model": model_name,
            "shrink": shrink,
            "train_seconds": train_seconds,
            "val_FIT_sim": val_metrics["FIT"],
            "val_RMSE_sim": val_metrics["RMSE"],
            "test_FIT_sim": test_metrics["FIT"],
            "test_RMSE_sim": test_metrics["RMSE"],
            "test_Bias_sim": test_metrics["Bias"],
        })

search_df = pd.DataFrame(rows).sort_values(["val_FIT_sim", "test_FIT_sim"], ascending=[False, False]).reset_index(drop=True)
best_by_validation = search_df.iloc[0].to_dict()
search_df.head(20).round(4)


,model,shrink,train_seconds,val_FIT_sim,val_RMSE_sim,test_FIT_sim,test_RMSE_sim,test_Bias_sim
0,random_forest_leaf5,0.8,27.1406,71.2298,0.8638,69.4550,0.8898,-0.0001
1,random_forest_leaf5,1.0,27.1406,71.1733,0.8655,69.4381,0.8903,-0.0059
2,random_forest_leaf5,0.7,27.1406,71.1625,0.8658,69.3628,0.8924,0.0024
3,hgb_31,1.0,2.6572,71.0813,0.8682,69.2859,0.8947,-0.0405
4,hgb_31,0.8,2.6572,71.0573,0.8690,69.2163,0.8967,-0.0280
5,random_forest_leaf5,0.6,27.1406,71.0308,0.8698,69.2026,0.8971,0.0048
6,extra_trees_leaf5,0.8,6.2214,70.9802,0.8713,69.3191,0.8937,-0.0042
7,hgb_15_l2,1.0,0.9064,70.9699,0.8716,69.0952,0.9002,-0.0470
8,hgb_31,0.7,2.6572,70.9670,0.8717,69.1030,0.9000,-0.0220
9,extra_trees_leaf5,0.7,6.2214,70.9240,0.8730,69.2335,0.8962,-0.0012


## 4. Compare


In [5]:
comparison_rows = [
    {
        "model": "ARX Search V1 best",
        "val_FIT_sim": arx_val_metrics["FIT"],
        "test_FIT_sim": arx_test_metrics["FIT"],
        "test_RMSE_sim": arx_test_metrics["RMSE"],
        "test_gain_vs_arx": 0.0,
    },
    {
        "model": "Hybrid Residual V1 best-by-val",
        "val_FIT_sim": best_by_validation["val_FIT_sim"],
        "test_FIT_sim": best_by_validation["test_FIT_sim"],
        "test_RMSE_sim": best_by_validation["test_RMSE_sim"],
        "test_gain_vs_arx": best_by_validation["test_FIT_sim"] - arx_test_metrics["FIT"],
    },
]

narx_v6_path = PROJECT_ROOT / "NARX" / "narx_v6.json"
if narx_v6_path.exists():
    with narx_v6_path.open("r", encoding="utf-8") as f:
        narx_v6 = json.load(f)
    selected = narx_v6["selected_candidate"]
    comparison_rows.append({
        "model": "NARX V6 selected",
        "val_FIT_sim": selected["val_FIT_sim"],
        "test_FIT_sim": selected["test_FIT_sim"],
        "test_RMSE_sim": selected["test_RMSE_sim"],
        "test_gain_vs_arx": selected["test_FIT_sim"] - arx_test_metrics["FIT"],
    })

comparison_df = pd.DataFrame(comparison_rows).sort_values("test_FIT_sim", ascending=False).reset_index(drop=True)
comparison_df.round(4)


,model,val_FIT_sim,test_FIT_sim,test_RMSE_sim,test_gain_vs_arx
0,Hybrid Residual V1 best-by-val,71.2298,69.4550,0.8898,2.6213
1,NARX V6 selected,70.1837,68.5310,0.9166,1.6973
2,ARX Search V1 best,68.9559,66.8337,0.9661,0.0000


## 5. Save


In [6]:
def json_ready(value):
    if isinstance(value, dict):
        return {str(k): json_ready(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_ready(v) for v in value]
    if isinstance(value, np.ndarray):
        return json_ready(value.tolist())
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if isinstance(value, Path):
        return str(value)
    return value


def df_to_markdown(df: pd.DataFrame) -> str:
    df_str = df.astype(str)
    headers = list(df_str.columns)
    lines = [
        "| " + " | ".join(headers) + " |",
        "| " + " | ".join(["---"] * len(headers)) + " |",
    ]
    for _, row in df_str.iterrows():
        lines.append("| " + " | ".join(str(row[col]) for col in headers) + " |")
    return "\n".join(lines) + "\n"


artifact = {
    "model_type": "Hybrid_ARX_Residual_NARX",
    "version": "hybrid_residual_v1",
    "backbone": {
        "model": "ARX",
        "order": "(1,5,2)",
        "na": 1,
        "nb": 5,
        "nk": 2,
        "include_intercept": True,
        "estimator": "OLS",
    },
    "residual_target": "y_true_sim - y_arx_sim",
    "selection_metric": "validation FIT_sim",
    "best_by_validation": best_by_validation,
    "arx_metrics": {
        "validation": arx_val_metrics,
        "test": arx_test_metrics,
    },
    "search_results": search_df.to_dict(orient="records"),
    "comparison": comparison_df.to_dict(orient="records"),
}

OUT_DIR.mkdir(exist_ok=True)
json_path = OUT_DIR / "hybrid_residual_v1.json"
csv_path = OUT_DIR / "hybrid_residual_v1_search.csv"
comparison_csv_path = OUT_DIR / "hybrid_residual_v1_comparison.csv"
comparison_md_path = OUT_DIR / "hybrid_residual_v1_comparison.md"

with json_path.open("w", encoding="utf-8") as f:
    json.dump(json_ready(artifact), f, indent=2)
    f.write("\n")

search_df.to_csv(csv_path, index=False)
comparison_df.to_csv(comparison_csv_path, index=False)
comparison_md_path.write_text(df_to_markdown(comparison_df.round(4)), encoding="utf-8")

json_path, csv_path, comparison_md_path


(WindowsPath('C:/Users/minht/OneDrive/Desktop/ARX-Model/Hybrid_ARX_NARX/hybrid_residual_v1.json'),
 WindowsPath('C:/Users/minht/OneDrive/Desktop/ARX-Model/Hybrid_ARX_NARX/hybrid_residual_v1_search.csv'),
 WindowsPath('C:/Users/minht/OneDrive/Desktop/ARX-Model/Hybrid_ARX_NARX/hybrid_residual_v1_comparison.md'))